## Data Quality Analysis (Missing, Duplicates and Data Types fixes)

In [55]:
import pandas as pd
import numpy as np
from configs.settings import project_dir

In [56]:
p_dir = project_dir()

#path to raw data files
raw_data_dir = p_dir.RAW_DATA_DIR
customers = raw_data_dir/'olist_customers_dataset.csv'
location = raw_data_dir/'olist_geolocation_dataset.csv'
items = raw_data_dir/'olist_order_items_dataset.csv'
payments = raw_data_dir/'olist_order_payments_dataset.csv'
reviews = raw_data_dir/'olist_order_reviews_dataset.csv'
orders = raw_data_dir/'olist_orders_dataset.csv'
products = raw_data_dir/'olist_products_dataset.csv'
sellers = raw_data_dir/'olist_sellers_dataset.csv'
category = raw_data_dir /'product_category_name_translation.csv'

In [57]:
# clean data dir
clean_data_dir = p_dir.CLEANED_DIR

### Customers Dataset 

In [110]:
customers_df = pd.read_csv(customers)
customers_df.sample(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
28508,93be387010f0b00896cf0d40d5921885,2547d56bb668d0543d4902cf3f2e7515,2043,sao paulo,SP
11889,44ad230ebeb51b5af647b688304f7bfc,5749c40b17909adc8c37d8f414ad2e52,2274,sao paulo,SP
47079,861967030fce9a8f82fa9f91127c3213,db9e4f0291db7866afc9956062d22fdd,7600,mairipora,SP
64894,b2cbce849410403be11de7f51dc04ac0,7bf5c9596fc39d41078cb46a86b23831,22260,rio de janeiro,RJ
53291,e705e099e9e08d5b78c6416381dd0728,a9004f180c4a46d9bbf6790a2fc2a3be,30770,belo horizonte,MG


In [59]:
# Missing Values
print(customers_df.isnull().sum())
print(f"\n\nNumber of rows with duplicates rows: {customers_df.duplicated().sum()}\n\n")
customers_df.info()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


Number of rows with duplicates rows: 0


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [60]:
customers_df['customer_city'].value_counts().to_string('../arson/customers_city.csv')

In [61]:
print(f"Number of duplicates value in customer_city col is {customers_df['customer_city'].unique().duplicated().sum()}\n")
print(f"Number of duplicates values in customer state is {customers_df['customer_state'].value_counts().duplicated().sum()}")

customers_df['customer_zip_code_prefix'].value_counts().sort_values(ascending=False).to_string('../arson/zip_code.txt')

Number of duplicates value in customer_city col is 0

Number of duplicates values in customer state is 0


In [62]:
customers_df.to_csv(clean_data_dir/'clean_customers.csv')

### Geolocation Dataset

In [63]:
location_df = pd.read_csv(location)
location_df.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [64]:
location_df.shape

(1000163, 5)

In [65]:
# print(location_df.isnull().sum())
# print(f"Number of Duplicates in location dataset of {location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/location_city.txt')}")

location_df['geolocation_city'] = location_df['geolocation_city'].astype('string')
location_df['geolocation_city'] = location_df['geolocation_city'].str.strip()
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('`','',regex='False')
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('^','',regex='False')
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('~','',regex='False')
# location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/location_city_clean.txt')

In [66]:
location_df[['geolocation_lat','geolocation_lng']].duplicated().sum()

np.int64(281700)

In [67]:
location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/lat.csv')


In [68]:
location_df.head()
location_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  string 
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(1), string(1)
memory usage: 38.2 MB


In [69]:
messy_data = location_df[location_df[['geolocation_lat','geolocation_lng']].duplicated()]
# messy_data[['geolocation_city','geolocation_lat','geolocation_lng']].value_counts().head(30)
messy_data['geolocation_city'] = messy_data['geolocation_city'].astype(str)
messy_data['geolocation_city'] = messy_data['geolocation_city'].str.strip()
messy_data[['geolocation_lat','geolocation_lng','geolocation_city']].groupby(['geolocation_lat','geolocation_lng']).value_counts().sort_values(ascending=False).to_string('../arson/messy1.csv')

In [70]:
messy_data[['geolocation_lat','geolocation_lng']].duplicated()

15         False
44          True
65          True
66         False
67          True
           ...  
1000153     True
1000154     True
1000159    False
1000160    False
1000162     True
Length: 281700, dtype: bool

In [71]:
messy_data[messy_data['geolocation_city'] == 'xambre']

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
897964,87535,-23.729684,-53.489767,xambre,PR
898303,87535,-23.729684,-53.489767,xambre,PR
898430,87535,-23.733100,-53.488682,xambre,PR
898450,87535,-23.737584,-53.485975,xambre,PR


In [72]:
messy_data[['geolocation_city','geolocation_lat','geolocation_lng']][messy_data[['geolocation_lat','geolocation_lng']].duplicated()].head(30).sort_values(by=['geolocation_lat','geolocation_lng'])

,geolocation_city,geolocation_lat,geolocation_lng
237,sao paulo,-23.552496,-46.632060
337,sao paulo,-23.552235,-46.628441
136,sao paulo,-23.549854,-46.643139
161,sao paulo,-23.549854,-46.643139
223,sao paulo,-23.549854,-46.643139
240,sao paulo,-23.549854,-46.643139
280,sao paulo,-23.549819,-46.635606
253,sao paulo,-23.546935,-46.636588
275,sao paulo,-23.546935,-46.636588
306,são paulo,-23.546935,-46.636588


In [73]:
messy_data[['geolocation_lat','geolocation_lng','geolocation_city']].where(messy_data[['geolocation_lat','geolocation_lng']].duplicated())

,geolocation_lat,geolocation_lng,geolocation_city
15,NaN,NaN,NaN
44,-23.546081,-46.644820,sao paulo
65,-23.546081,-46.644820,sao paulo
66,NaN,NaN,NaN
67,-23.546081,-46.644820,sao paulo
...,...,...,...
1000153,-28.343273,-51.873734,ciriaco
1000154,-28.070493,-52.011342,tapejara
1000159,NaN,NaN,NaN
1000160,NaN,NaN,NaN


In [74]:
messy_data[messy_data['geolocation_zip_code_prefix'].duplicated()]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
72,1046,-23.545320,-46.644069,sao paulo,SP
82,1046,-23.546081,-46.644820,sao paulo,SP
...,...,...,...,...,...
1000153,99970,-28.343273,-51.873734,ciriaco,RS
1000154,99950,-28.070493,-52.011342,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS


In [75]:
location_df[location_df['geolocation_zip_code_prefix'].duplicated()].sort_values(by='geolocation_zip_code_prefix').head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1384,1001,-23.549292,-46.633559,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
1351,1001,-23.549951,-46.634027,são paulo,SP
235,1001,-23.550642,-46.634410,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP
1004,1001,-23.549292,-46.633559,sao paulo,SP
575,1001,-23.549779,-46.633957,são paulo,SP
519,1001,-23.551337,-46.634027,sao paulo,SP
1062,1001,-23.550498,-46.634338,sao paulo,SP
299,1001,-23.549698,-46.633909,sao paulo,SP


In [76]:
location_df_clean = location_df.drop_duplicates(subset=['geolocation_lat','geolocation_lng'])

In [77]:
print(location_df.shape)
print(location_df_clean.shape)

(1000163, 5)
(718463, 5)


In [78]:
1000163 - 718463

281700

In [79]:
location_df.duplicated().sum()

np.int64(261831)

In [80]:
location_df_clean.duplicated().sum()

np.int64(0)

In [81]:
# Saving cleaned dataset
location_df_clean.to_csv(clean_data_dir/'cleaned_location.csv')

## Order Items dataset cleaning

In [82]:
order_items_df = pd.read_csv(items)
order_items_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [83]:
order_items_df.info()
# No null values, only incorrect dtype of date column

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [84]:
order_items_df['order_item_id'].value_counts()
# order_items_df.duplicated().sum()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

In [85]:
# fixing the dtype of date col
order_items_df['shipping_limit_date'] = order_items_df['shipping_limit_date'].astype('datetime64[ns]')

In [86]:
order_items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [87]:
order_items_df.to_csv(clean_data_dir/'cleaned_orders_items.csv')

### Order Payments Dataset cleaning

In [88]:
order_payments_df = pd.read_csv(payments)
order_payments_df.sample(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
67064,06613112cdd7761776322081835b1b96,1,credit_card,2,44.35
16448,68873cf91053cd11e6b49a766db5af1a,1,boleto,1,91.66
80608,a07b30a459f83015fb2927d732e20943,1,credit_card,1,118.02
18744,06addb9b5cc9c47e661a6eed12276418,1,credit_card,4,260.42
9838,86274c00b0ec7c91b000d4c6cac1dc72,1,credit_card,5,129.84
44075,8282d5532e3d6bc650960861965714ee,1,credit_card,6,64.00
20606,202bcf28f7e3dfcee06ab0bfe10c750e,1,credit_card,1,58.37
18329,dd14b7426d230b6c50002b13efdcbc9f,1,credit_card,2,95.22
88414,b47f53f735ee76b0f07d10b4525d564e,1,credit_card,4,244.85
40918,0d40f8fd78977af4c4fa84293ae7fa21,1,credit_card,4,221.79


In [89]:
# order_payments_df.info()
# order_payments_df['payment_type'].value_counts()
order_payments_df.groupby('payment_type')['payment_installments'].value_counts()

payment_type  payment_installments
boleto        1                       19784
credit_card   1                       25455
              2                       12413
              3                       10461
              4                        7098
              10                       5328
              5                        5239
              8                        4268
              6                        3920
              7                        1626
              9                         644
              12                        133
              15                         74
              18                         27
              11                         23
              24                         18
              20                         17
              13                         16
              14                         15
              17                          8
              16                          5
              21                         

In [90]:
order_payments_df.duplicated().sum()

np.int64(0)

In [91]:
## replace boleto with wallet, handled not_defined values with median
order_payments_df['payment_type'] = order_payments_df['payment_type'].str.replace('boleto','wallet')

order_payments_df['payment_type'] = order_payments_df['payment_type'].str.replace('not_defined',order_payments_df['payment_type'].mode()[0])

In [92]:
order_payments_df['payment_type'].value_counts()

payment_type
credit_card    76798
wallet         19784
voucher         5775
debit_card      1529
Name: count, dtype: int64

In [93]:
order_payments_df.to_csv(clean_data_dir/'cleaned_order_payments.csv')

## Order Reviews Dataset cleaning

In [94]:
order_reviews_df = pd.read_csv(reviews)
order_reviews_df.sample(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
8670,90289830e1053f11c8d085a128ab5bb9,58e7140d9d368a70076dc999a7a7bf85,1,NaN,Não recebi todo os produto?,2017-11-28 00:00:00,2017-11-28 15:44:28
15843,d65cc70adf279ab2cf262d3b1391cf9c,2d3246c68cbe85d64b652f4ff3848de2,5,NaN,MUITO BOA A COMPRA,2017-03-16 00:00:00,2017-03-23 17:59:14
17098,9f8a41b6506a8395e1d9f9084dc7f83e,4c3bec2f85aac1c764e1597247d92b0e,5,NaN,NaN,2018-04-11 00:00:00,2018-04-14 14:11:11
80575,5d82901c5ae1b1c099e38a921eba4d6e,cb269e12d603437a1a1a1bf4c27c3716,5,NaN,NaN,2018-02-28 00:00:00,2018-03-01 14:24:07
85608,dfa40198a6eac9daf153af122d0832cb,81039bb5e28d76cd6757558d5800a617,4,NaN,NaN,2017-10-11 00:00:00,2017-10-14 05:05:05
8513,f60e942a213a57b16dee10b28b800468,8a7f0e6d9767d52a05dd48229e24d766,2,NaN,Estou aguardando instalação do produto em loja...,2017-02-04 00:00:00,2017-02-06 15:52:04
53074,163a3dcaf0954acca9d623e61f3ab97c,666ee8dddb0138c7703e3e2a8bb8919f,3,NaN,NaN,2018-05-16 00:00:00,2018-05-16 22:43:24
69156,546c5fc7a406f1734a427d1c3eee4745,2b3c116f4ae22a4b4a127c73d39f5d6e,5,NaN,Muito bom os produtos,2017-07-01 00:00:00,2017-07-02 18:23:31
3843,9e1dff28e43a8e616e517531e72821b7,17be7ed9263bf0ea1f2e8ad91f699dac,4,NaN,Gostei do serviço e prazo de entrega,2018-08-21 00:00:00,2018-08-22 21:41:19
26917,eabfeafecab228244b0ac7a713236d0e,92ebaccb732e2b347a19dbc3245fcb27,2,NaN,Pelo valor dele eu esperava um produto melhor.,2017-12-28 00:00:00,2017-12-29 00:36:20


In [95]:
order_reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [96]:
order_reviews_df.duplicated().sum()
order_reviews_df['review_score'].value_counts()
# TO-DO
## handle the date dtype col, handle missing value in both title and comment message.(a better approach to handle, cant drop all these rows)

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

In [97]:
# fixing the date time dtype col
order_reviews_df['review_creation_date'] = order_reviews_df['review_creation_date'].astype('datetime64[ns]')
order_reviews_df['review_answer_timestamp'] = order_reviews_df['review_answer_timestamp'].astype('datetime64[ns]')

In [98]:
print(f"duplicate review id in table is {order_reviews_df['review_id'].duplicated().sum()}")
order_reviews_df = order_reviews_df.drop_duplicates(subset='review_id',keep='last')
# order_reviews_df.groupby('review_id')[['review_comment_title','review_comment_message']].value_counts().to_string('../arson/review_id.csv')
order_reviews_df.shape

duplicate review id in table is 814


(98410, 7)

In [99]:
print(order_reviews_df.isnull().sum())
## TO-DO: need to analysis these missing values and an approach to handle these values



review_id                      0
order_id                       0
review_score                   0
review_comment_title       86891
review_comment_message     57742
review_creation_date           0
review_answer_timestamp        0
dtype: int64


In [100]:
order_reviews_df[['review_id','review_comment_title','review_comment_message']].sample(20)
# checking whether

,review_id,review_comment_title,review_comment_message
3004,bffba33345820ccfaf1f783a1d577ab1,NaN,NaN
37139,dc5410871c5f81c3f4a91f42c500952c,NaN,NaN
72786,df3bde559138dc2ec2d4340e8cfbef26,NaN,NaN
75791,909d528441ba3533275860a47f39ee63,NaN,NaN
16413,ea710f5b8029ed7a4973ec47402a24de,NaN,NaN
18118,2d66f22841dcfd16f9e4accaca2c6e42,NaN,Sempre comprei nesta loja. Recomendo com certeza.
51921,48ec5aadfaa27b0cab76d019eb713da6,Tudo perfeito,Tudo perfeito celular maravilhoso super recome...
48115,5f78ff9826cadab6b7550a1718d1b978,Muito satisfeito,NaN
63978,f597a9739b74e939b29745f4657f32f9,NaN,NaN
1643,ef8aa48087495ca0ecf32e3375e09d45,BOM,"De forma geral o produto é bom, porém, som sai..."


In [101]:

order_reviews_df[order_reviews_df['review_comment_message'].isna()][['review_id','review_comment_title','review_comment_message','review_score']]

,review_id,review_comment_title,review_comment_message,review_score
0,7bc2406110b926393aa56f80a40eba40,NaN,NaN,4
1,80e641a11e56f04c1ad469d5645fdfde,NaN,NaN,5
2,228ce5500dc1d8e020d8d1322874b6f0,NaN,NaN,5
5,15197aa66ff4d0650b5434f1b46cda19,NaN,NaN,1
6,07f9bee5d1b850860defd761afa7ff16,NaN,NaN,5
...,...,...,...,...
99217,c6b270c61f67c9f7cb07d84ea8aeaf8b,NaN,NaN,5
99218,af2dc0519de6e0720ef0c74292fb4114,NaN,NaN,5
99219,574ed12dd733e5fa530cfd4bbf39d7c9,NaN,NaN,5
99220,f3897127253a9592a73be9bdfdf4ed7a,NaN,NaN,5


In [102]:
## filling title and message with no comment
order_reviews_df[['review_comment_title','review_comment_message']] = order_reviews_df[['review_comment_title','review_comment_message']].fillna('No Comment')

In [103]:
# creating a new col has_commnet or not
order_reviews_df['has_comment'] = (order_reviews_df['review_comment_title'] != 'No Comments')

In [104]:
order_reviews_df.sample(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_comment
44078,dc0780f54e22f76563bc3c6bb7d6f78a,ac1dce035adc8c31bd95b3042286e137,3,No Comment,No Comment,2017-04-07,2017-04-09 04:56:11,True
4969,7a079a0993939a7ebdc7868c3358902c,faa068942b581bc31538abdea52e6f35,5,No Comment,No Comment,2018-05-08,2018-05-10 15:57:19,True
9705,45fe1f264beed651c351e834b984bf31,ce19416583ee035a7a8097db2c53f98c,5,No Comment,No Comment,2017-09-13,2017-09-19 15:25:41,True
11359,7c9fc5ab905aac3830d9edefb42afd32,aff4e35a92033577c37cf3a99f640cd9,5,No Comment,No Comment,2018-03-30,2018-04-02 12:54:08,True
38652,71b3d30584763a24bc52690b83c93a14,c2f068ae849d057bf039aca384d2591d,4,No Comment,bom,2018-03-06,2018-03-08 16:00:35,True
56678,355ab6d221abdfb4c02e27695fcc6d5f,0652543c9f8586d33500c3d0e4ddc6a6,4,No Comment,"Superou as minhas expectativas, entregue antes...",2017-05-19,2017-05-20 13:00:35,True
14777,41a15c1d298e2bc5c23ec0dac4814d2f,5ff8bece1ba3adcb585b884358bb9917,4,No Comment,No Comment,2018-08-18,2018-08-18 10:50:11,True
60129,3a44f7ffe0e2d9073c83cea3d86807b8,1fec6937fb6666c83cc1d3078a1949ab,3,No Comment,No Comment,2018-06-06,2018-06-06 20:17:52,True
56942,1b31adb1d6129b0faef872ecffd06a74,2c9692d95147996e657865c7f5ec42fc,5,No Comment,"a briga em casa pelo travesseiro, preciso de m...",2017-12-09,2017-12-09 21:24:50,True
1888,f5b929ba271b92b16c293dc2b8a6c881,4f405b650a39270ace3d575d308884d4,4,No Comment,No Comment,2018-01-25,2018-01-30 14:48:59,True


In [105]:
order_reviews_df.isnull().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
has_comment                0
dtype: int64

In [106]:
order_reviews_df[['review_comment_title','review_comment_message']].value_counts().to_string('../arson/comments.csv')

In [107]:
order_reviews_df.to_csv(clean_data_dir/'cleaned_order_reviews.csv')